# Algoritmos da Disciplina: congress-Twitter (G24)

## Verificar o ambiente

In [14]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\lucas\OneDrive\Área de Trabalho\analise-snap\.venv\Scripts\python.exe
3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]


## Carregar o grafo

In [15]:
import math
import networkx as nx

caminho = "dados/congress_network/congress.edgelist"
G = nx.read_edgelist(caminho, create_using=nx.DiGraph, nodetype=int)  # grafo dirigido
n = G.number_of_nodes()

# Custo de cada aresta: -log(probabilidade). Maior influência = menor custo.
for u, v, d in G.edges(data=True):
    d["custo"] = -math.log(d["weight"])

# Versão não dirigida: A->B e B->A viram uma aresta só, com a maior das duas probabilidades.
G_nd = nx.Graph()
G_nd.add_nodes_from(G.nodes())
for u, v, d in G.edges(data=True):
    p = d["weight"]
    if G_nd.has_edge(u, v):
        p = max(p, G_nd[u][v]["weight"])
    G_nd.add_edge(u, v, weight=p, custo=-math.log(p))

print(f"Vértices: {n}, arestas (dirigido): {G.number_of_edges()}, arestas (não dirigido): {G_nd.number_of_edges()}")

Vértices: 475, arestas (dirigido): 13289, arestas (não dirigido): 10222


## Medição de tempo

In [16]:
import time
import statistics
from scipy import stats

tempos = {}

def medir_tempo(funcao, repeticoes=30):
    lista = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        funcao()
        lista.append(time.perf_counter() - inicio)
    return lista

def mostrar_tempos(lista, colunas=10):
    print("Tempos (ms):")
    for i in range(0, len(lista), colunas):
        print("  ".join(f"{t * 1000:10.4f}" for t in lista[i:i + colunas]))

def resumir(lista):
    k = len(lista)
    media = statistics.mean(lista)
    desvio = statistics.stdev(lista)
    critico = 1.96 if k >= 30 else stats.t.ppf(0.975, k - 1)  # normal (k >= 30) ou t-Student
    margem = critico * desvio / k ** 0.5
    print(f"Média: {media * 1000:.4f} ms, desvio padrão: {desvio * 1000:.4f} ms")
    print(f"IC 95%: [{(media - margem) * 1000:.4f}; {(media + margem) * 1000:.4f}] ms")

def cronometrar(nome, funcao):
    tempos[nome] = medir_tempo(funcao)
    mostrar_tempos(tempos[nome])
    resumir(tempos[nome])

print("Funções de medição prontas.")

Funções de medição prontas.


## Busca em largura (BFS)

In [17]:
arestas = list(nx.bfs_edges(G, 0))
print(f"Alcança {len(arestas) + 1} vértices a partir do vértice 0.")

cronometrar("BFS", lambda: list(nx.bfs_edges(G, 0)))

Alcança 469 vértices a partir do vértice 0.
Tempos (ms):
    3.1475      2.0276      1.6482      1.9098      2.5343      1.7443      1.5868      1.5571      1.6990      1.5691
    2.5853      1.6382      2.0500      2.3212      2.0322      1.6124      1.6002      1.5612      1.7286      1.5674
    1.6608      2.1479      2.5200      2.1548      2.1479      2.1302      1.8087      1.5360      1.5290      1.8932
Média: 1.9216 ms, desvio padrão: 0.3965 ms
IC 95%: [1.7798; 2.0635] ms


## Busca em profundidade (DFS)

In [18]:
arestas = list(nx.dfs_edges(G, 0))
print(f"Alcança {len(arestas) + 1} vértices a partir do vértice 0.")

cronometrar("DFS", lambda: list(nx.dfs_edges(G, 0)))

Alcança 469 vértices a partir do vértice 0.
Tempos (ms):
    3.5719      2.2071      2.1146      2.1306      1.9958      2.2415      1.8853      1.3769      1.3090      1.4429
    1.8506      2.7870      2.2005      1.6728      2.8208      2.0667      1.7794      1.7152      1.9448      2.1503
    2.3148      2.2382      1.8683      1.6587      1.6585      1.7177      1.9021      1.8766      2.3052      3.4069
Média: 2.0737 ms, desvio padrão: 0.5198 ms
IC 95%: [1.8877; 2.2597] ms


## Verificação de eulerianidade

In [19]:
print(f"Dirigido: {'euleriano' if nx.is_eulerian(G) else 'não euleriano'}")
print(f"Não dirigido: {'euleriano' if nx.is_eulerian(G_nd) else 'não euleriano'}")

cronometrar("Eulerianidade", lambda: (nx.is_eulerian(G), nx.is_eulerian(G_nd)))

Dirigido: não euleriano
Não dirigido: não euleriano
Tempos (ms):
    0.0355      0.0157      0.0141      0.0124      0.0137      0.0124      0.0129      0.0143      0.0118      0.0111
    0.0110      0.0142      0.0110      0.0116      0.0121      0.0117      0.0114      0.0109      0.0129      0.0118
    0.0115      0.0126      0.0120      0.0133      0.0120      0.0119      0.0116      0.0114      0.0118      0.0112
Média: 0.0131 ms, desvio padrão: 0.0044 ms
IC 95%: [0.0115; 0.0146] ms


## Dijkstra

In [20]:
distancias = nx.single_source_dijkstra_path_length(G, 0, weight="custo")
print(f"Alcança {len(distancias)} vértices a partir do vértice 0.")
print(f"Maior custo: {max(distancias.values()):.4f}")

cronometrar("Dijkstra", lambda: nx.single_source_dijkstra_path_length(G, 0, weight="custo"))

Alcança 469 vértices a partir do vértice 0.
Maior custo: 21.9834
Tempos (ms):
    9.8305     10.7655     10.7725      8.3894      9.7503      7.5657      6.6787      7.2186      7.0102      7.8972
    6.4741      7.3411      7.5396     10.3576      9.2537      9.4557      8.4736      9.2582      7.4162      7.5945
    8.0583      7.8707      6.9982      7.2305      8.9759      8.8719      9.8385     10.9238      7.8598      9.6691
Média: 8.5113 ms, desvio padrão: 1.3156 ms
IC 95%: [8.0405; 8.9821] ms


## Bellman-Ford

In [21]:
distancias = nx.single_source_bellman_ford_path_length(G, 0, weight="custo")
print(f"Alcança {len(distancias)} vértices a partir do vértice 0.")
print(f"Maior custo: {max(distancias.values()):.4f}")

cronometrar("Bellman-Ford", lambda: nx.single_source_bellman_ford_path_length(G, 0, weight="custo"))

Alcança 469 vértices a partir do vértice 0.
Maior custo: 21.9834
Tempos (ms):
   12.2664     11.0256     10.1199     12.5739     12.7279     12.0358     11.0181     12.3216     12.2332     10.2285
   12.4694     15.0643     11.6486     14.1180     19.5995     14.7033     13.1725     13.3701     12.9303     11.5886
    9.5431     12.2108     12.4384     11.6036     13.6863     15.1175     14.9120     15.5791     14.2654     14.3881
Média: 12.9653 ms, desvio padrão: 2.0035 ms
IC 95%: [12.2484; 13.6823] ms


## Floyd-Warshall

In [22]:
import numpy as np

distancias = nx.floyd_warshall_numpy(G, nodelist=range(n), weight="custo")
print(f"Matriz de distâncias: {distancias.shape[0]} x {distancias.shape[1]}")
print(f"Maior custo: {distancias[np.isfinite(distancias)].max():.4f}")

cronometrar("Floyd-Warshall", lambda: nx.floyd_warshall_numpy(G, nodelist=range(n), weight="custo"))

Matriz de distâncias: 475 x 475
Maior custo: 32.1171
Tempos (ms):
 1474.4500   1034.7566   1038.1459   1111.3596   1053.2137   1077.1972   1104.3289   1043.1627   1110.7298   1023.6433
  970.9584    970.6120    952.3374   1333.0005    935.0196    929.1083    954.6689    948.8777    903.8617   1028.7647
  944.1699    944.6768    975.0835    930.0053   1056.7057    986.5303    951.5612   1053.2349   1135.2180   1005.6825
Média: 1032.7022 ms, desvio padrão: 120.3284 ms
IC 95%: [989.6432; 1075.7611] ms


## Algoritmo de Tarjan

In [23]:
componentes = sorted(nx.strongly_connected_components(G), key=len, reverse=True)
print(f"Componentes fortemente conexas: {len(componentes)}")
print(f"Tamanhos: {', '.join(str(len(x)) for x in componentes)}")

cronometrar("Tarjan", lambda: list(nx.strongly_connected_components(G)))

Componentes fortemente conexas: 7
Tamanhos: 469, 1, 1, 1, 1, 1, 1
Tempos (ms):
    9.7596      8.5070      8.4646     10.9624     10.7827      9.1268      8.2709     10.0827     11.1971      9.9035
   10.8763      8.8702      9.5307     11.5083      9.3673     10.7912      9.8208      8.3331      7.7762      8.1736
    7.4054     11.6022      7.1335      9.5057      7.7619     10.8233      8.9442      9.2189      8.7892      8.8149
Média: 9.4035 ms, desvio padrão: 1.2513 ms
IC 95%: [8.9557; 9.8513] ms


## Árvore geradora mínima (Kruskal)

In [24]:
arvore = nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="kruskal")
print(f"Arestas na árvore: {arvore.number_of_edges()}")
print(f"Custo total: {arvore.size(weight='custo'):.4f}")

cronometrar("Kruskal", lambda: nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="kruskal"))

Arestas na árvore: 474
Custo total: 1748.5905
Tempos (ms):
   28.2646     30.0740     33.2597     35.9033     31.6755     90.3051     44.6560     31.0813     34.5774     30.1766
   82.8496     29.9975     31.4684     29.9424     37.4222     30.2253     28.3563     29.3119     29.7573     28.8314
   80.2590     33.3592     35.6546     32.4297     29.9026     36.5742     29.4153     36.3167     30.8360     26.4109
Média: 37.3098 ms, desvio padrão: 16.4517 ms
IC 95%: [31.4226; 43.1970] ms


## Árvore geradora mínima (Prim)

In [25]:
arvore = nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="prim")
print(f"Arestas na árvore: {arvore.number_of_edges()}")
print(f"Custo total: {arvore.size(weight='custo'):.4f}")

cronometrar("Prim", lambda: nx.minimum_spanning_tree(G_nd, weight="custo", algorithm="prim"))

Arestas na árvore: 474
Custo total: 1748.5905
Tempos (ms):
   27.0313     22.8515     19.4822     19.2673     18.9189     22.4630     25.4195     22.5018     21.1056     28.1436
   80.5302     25.6145     26.4180     24.5965     24.0193     26.3609     27.1260     31.5480     24.2250     20.3049
   22.6352     71.3392     20.4081     25.3144     19.3574     20.0107     21.0612     19.0554     19.9038     20.9452
Média: 26.5986 ms, desvio padrão: 13.8382 ms
IC 95%: [21.6467; 31.5506] ms


## Exportar tempos

In [26]:
import csv
import os

if not tempos:
    print("Nenhum tempo medido ainda: rode as seções dos algoritmos antes de exportar.")
else:
    os.makedirs("resultados", exist_ok=True)
    arquivo = "resultados/tempos_algoritmos.csv"
    with open(arquivo, "w", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f)
        escritor.writerow(["algoritmo", "execucao", "tempo_ms"])
        for algoritmo, lista in tempos.items():
            for i, t in enumerate(lista, start=1):
                escritor.writerow([algoritmo, i, f"{t * 1000:.4f}"])
    with open("resultados/stats_algoritmos.csv", "w", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f)
        escritor.writerow(["algoritmo", "n", "media_ms", "desvio_ms", "ic95_inf_ms", "ic95_sup_ms"])
        for algoritmo, lista in tempos.items():
            k = len(lista)
            media = statistics.mean(lista) * 1000
            desvio = statistics.stdev(lista) * 1000
            critico = 1.96 if k >= 30 else stats.t.ppf(0.975, k - 1)
            margem = critico * desvio / k ** 0.5
            escritor.writerow([algoritmo, k, f"{media:.4f}", f"{desvio:.4f}", f"{media - margem:.4f}", f"{media + margem:.4f}"])

    print(f"Algoritmos exportados: {', '.join(tempos)}")
    print(f"{sum(len(l) for l in tempos.values())} tempos salvos em {arquivo}")
    print("Resumo (média, desvio padrão e IC 95%) salvo em resultados/stats_algoritmos.csv")

Algoritmos exportados: BFS, DFS, Eulerianidade, Dijkstra, Bellman-Ford, Floyd-Warshall, Tarjan, Kruskal, Prim
270 tempos salvos em resultados/tempos_algoritmos.csv
Resumo (média, desvio padrão e IC 95%) salvo em resultados/stats_algoritmos.csv
